# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.


In [ ]:
# =========================================================
# CLOUD ENVIRONMENT SETUP (AUTO-GENERATED)
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print("Running in Cloud Environment")
    
    
    
    # Copy dataset
    print("Copying dataset...")
    !apt -qq install rclone && rclone copy /kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/ /kaggle/working/ --transfers 16 --checkers 16 --progress --ignore-existing -q --exclude "dataset/**" --checksum
    
    # Install dependencies
    print("Installing dependencies (this may take a minute)...")
    %pip install -r requirements.txt
    
    print("Setup Complete!")
else:
    print("Running locally. No setup needed.")


# 3D Human Motion Generation Pipeline

This notebook implements a complete pipeline for generating 3D human motion animations using:

- **Motion History Encoder**: Encodes motion context with text conditioning
- **Flow Matching Predictor**: Generates motion using flow matching
- **CLIP Text Encoder**: Encodes text prompts for conditioning

**Input**: Text description + optional motion history
**Output**: 3D joint positions (22 joints × 3D coordinates)


## Setup and Imports


In [ ]:
# Install dependencies (uncomment if running on Kaggle/Colab)
# !pip install torch transformers matplotlib tqdm

import torch
import torch.serialization  # ADD THIS
import pathlib
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from config import Config
from models import MotionHistoryEncoder, FlowMatchingPredictor, HumanMotionGenerator
from utils.dataset import Text2MotionDataset, text2motion_collate_fn, create_dataloader
from utils.text_encoder import CLIPEncoder
from utils.visualization import visualize_motion
from utils.motion_utils import FeatureNormalizer

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configuration
config = Config()
config.device = device
print(f"Dataset path: {config.dataset_path}")
print(f"Checkpoint dir: {config.checkpoint_dir}")

In [ ]:
# Create dataset and dataloader
print("Loading dataset...")
config.dataset_path = Path(
    "/kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/dataset/humanml3d-subset/"
)
config.checkpoint_dir = Path(
    "/kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/checkpoints/"
)

config.batch_size = 1
dataloader, normalizer = create_dataloader(config, split="train", shuffle=True)

print(f"Number of batches: {len(dataloader)}")

# Show a sample batch
sample_batch = next(iter(dataloader))
print(f"\nSample batch:")
print(f"  Captions: {len(sample_batch['captions'])} samples")
print(f"  Motion shape: {sample_batch['motion'].shape}")  # (B, T, 271)
print(f"  Joints shape: {sample_batch['joints'].shape}")  # (B, T, 22, 3)
print(f"  Text embeddings shape: {sample_batch['text_clip'].shape}")  # (B, 1, 512)
print(f"  Lengths shape: {sample_batch['lengths'].shape}")  # (B,)
print(f"\nSample caption: '{sample_batch['captions'][0]}'")

## Load Trained Model for Generation


In [ ]:
!ls -lah checkpoints

In [ ]:
torch.serialization.add_safe_globals([Config, pathlib.PosixPath, slice])

In [ ]:
# Load the best checkpoint for generation
checkpoint_path = config.checkpoint_dir / "best.pt"

generator = HumanMotionGenerator.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    config=config,
    device=str(device),
    normalizer=normalizer,
)

print(f"Loaded model from {checkpoint_path}")
print("Model ready for generation!")

## Initialize CLIP Text Encoder


In [ ]:
# Initialize CLIP encoder for text-to-embedding conversion
clip_encoder = CLIPEncoder()  # Default model is openai/clip-vit-base-patch32

print("CLIP encoder initialized")

## Generate Motion from Text Prompts

Generate 3D human motion animations from text descriptions.


In [ ]:
dataset_iter = iter(dataloader)
d = next(dataset_iter)
d["captions"][0], d["lengths"][0]

In [ ]:
motions_np = d["joints"][0, :].cpu().numpy()
ani = visualize_motion(
    motions_np, title=f"{d['captions'][0]} - {d['lengths'][0]}", fps=20
)
ani

In [ ]:
# Enter your custom text prompt
print(f"Generating motion for: '{d['captions']}'")
input_positions = d["joints"][:, 0:20, :, :]

# Use strict dataset CLIP embedding transport shape (B, 1, 512)
with torch.no_grad():
    text_embedding = d["text_clip"]

position_history, feature_history, prev_relative_shifts = generator.generate_sequence(
    text=text_embedding,
    input_positions=input_positions,
    num_frames=10,
    num_steps=100,
    guidance_scale=1,
)

motions_np = position_history[0, :].cpu().numpy()
ani = visualize_motion(motions_np, title=d["captions"][0], fps=20)
ani

In [ ]:
# Define text prompts for generation
text_prompts = [
    "a person walks forward",
    "a person is running",
    "a person jumps up",
]

# Generation parameters
num_steps = 25  # Number of flow matching steps (higher = better quality, slower)
guidance_scale = 2.5  # Kept for API parity with model call

print(f"Generating {len(text_prompts)} motions...")
print(f"Flow matching steps: {num_steps}")
print(f"Guidance scale: {guidance_scale}\n")

generated_motions = []

for i, prompt in enumerate(text_prompts):
    print(f"[{i+1}/{len(text_prompts)}] Generating: '{prompt}'")

    position_history, feature_history, prev_relative_shifts = (
        generator.generate_sequence(
            text=prompt,
            num_frames=100,
            num_steps=10,
            guidance_scale=2.5,
        )
    )

    generated_motions.append(position_history.cpu().numpy())
    print(f"  Generated shape: {position_history.shape}\n")

print("All motions generated successfully!")

## Visualize Generated Motions

Visualize each generated motion as an animated 3D skeleton.


In [ ]:
# Visualize first motion: "a person walks forward"
motion_idx = 0
joints = generated_motions[motion_idx][0]  # Remove batch dimension
prompt = text_prompts[motion_idx]

print(f"Visualizing: '{prompt}'")
print(f"Motion shape: {joints.shape}")

ani = visualize_motion(joints, title=prompt, fps=20)
ani

In [ ]:
# Visualize second motion: "a person is running"
motion_idx = 1
joints = generated_motions[motion_idx][0]
prompt = text_prompts[motion_idx]

print(f"Visualizing: '{prompt}'")
ani = visualize_motion(joints, title=prompt, fps=20)
ani

In [ ]:
# # Visualize third motion: "a person jumps up"
# motion_idx = 2
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

In [ ]:
# # Visualize fourth motion: "a person waves their hand"
# motion_idx = 3
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

In [ ]:
# # Visualize fifth motion: "a person sits down"
# motion_idx = 4
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

## Save Generated Motions


In [ ]:
# # Create output directory
# output_dir = config.output_path / "generated_motions"
# output_dir.mkdir(parents=True, exist_ok=True)

# # Save each generated motion
# for i, (motion, prompt) in enumerate(zip(generated_motions, text_prompts)):
#     # Save as numpy file
#     filename = f"motion_{i:03d}_{prompt.replace(' ', '_')[:30]}.npy"
#     filepath = output_dir / filename
#     np.save(filepath, motion)
#     print(f"Saved: {filepath}")

# print(f"\n✅ All motions saved to {output_dir}")

## Custom Text Prompt Generation

Try your own text prompts!


In [ ]:
# # Enter your custom text prompt
# custom_prompt = "a person does a backflip"  # Change this to your desired motion

# print(f"Generating motion for: '{custom_prompt}'")

# # Encode text to CLIP embedding
# motions = []
# with torch.no_grad():
#     text_embedding = clip_encoder([prompt]).to(device)  # (1, 512)

# joint_positions = generator.generate_sequence(
#     text=text_embedding,
#     num_frames=200,
#     num_steps=10,
#     guidance_scale=2.5,
#     dataset_type="t2m",
# )

# # Visualize
# motions_np = joint_positions.cpu().numpy()
# ani = visualize_motion(motions_np, title=custom_prompt, fps=20)
# ani